# From the stochastic equation to exponential durations

The article states that the durations of both states are exponential and reads the two
barriers off their means. This notebook asks what the model itself says: how the mean exit
time of a well grows as the noise falls, how close the classical Kramers formula and the
exact first passage time are to the times a simulated path actually spends, and when the exit
time is exponential at all. It ends with the Brownian bridge correction, which is what keeps
a crossing tested only at the grid points from running long.

## Provenance

**No clinical data are used in this notebook.** Every number below comes from the model: two
potentials, one calibrated to the two mean durations the article prints and one at the
asymmetry the article draws its own figures with, and paths integrated from them. The two
durations and the two parameters of the article are read from `msrelapse.PAPER`.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import msrelapse

# Every random step of this notebook is drawn from this one seed.
SEED = 20130910
# Paths behind each simulated point, and the integration step. A few thousand
# paths smooth the points at a cost that grows in proportion.
N_PATHS = 300
DT = 0.02

ALPHA = msrelapse.PAPER.alpha_reference.value

In [ ]:
calibrated_beta, calibrated_sigma = msrelapse.calibrate(
    msrelapse.PAPER.tau_health_cohort_weeks.value,
    msrelapse.PAPER.tau_no_health_cohort_weeks.value,
)
calibrated = msrelapse.DoubleWell(ALPHA, calibrated_beta)
illustrative = msrelapse.DoubleWell(ALPHA, msrelapse.PAPER.beta_illustrative.value)
wells = {
    "calibrated": calibrated,
    f"illustrative, beta = {illustrative.beta:g}": illustrative,
}

print(f"calibrated beta {calibrated_beta:.4f}, sigma {calibrated_sigma:.4f}")
for name, well in wells.items():
    print(name, well.barriers())

## The sweep

For each well, each side and each noise amplitude the sweep collects five numbers: the
Kramers escape time, the exact mean first passage time from the bottom of the well to the
saddle, the same from one well bottom to the other, and the mean and the coefficient of
variation of a few hundred simulated first passage times. The grid starts at 0.40 rather than
lower because the health well of the calibrated potential takes thousands of weeks to leave
below that, and simulating a few hundred such paths costs more than this notebook is meant to.

In [ ]:
SIGMAS = np.array([0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 0.80])

# One generator runs through the whole sweep, so that every point is a sample of
# its own rather than the same noise stream measured again at another sigma. The
# sweep is still reproducible, because the generator starts from the seed above.
sweep_generator = np.random.default_rng(SEED)
sweep_rows = []
for name, well in wells.items():
    for side in ("health", "relapse"):
        for sigma in SIGMAS:
            times = msrelapse.exit_times(
                well, sigma, side, N_PATHS, dt=DT, rng=sweep_generator, bridge_correction=True
            )
            bottoms = msrelapse.passage_endpoints(well, side, "bottom_to_bottom")
            sweep_rows.append(
                (
                    name,
                    side,
                    float(sigma),
                    msrelapse.kramers_time(well, sigma, side),
                    msrelapse.mfpt(well, sigma, side),
                    msrelapse.mfpt(well, sigma, side, *bottoms),
                    float(times.mean()),
                    float(times.std(ddof=1) / times.mean()),
                )
            )

sweep = pd.DataFrame(
    sweep_rows,
    columns=[
        "well",
        "side",
        "sigma",
        "kramers",
        "mfpt_bottom_to_saddle",
        "mfpt_bottom_to_bottom",
        "simulated_mean",
        "simulated_cv",
    ],
)
sweep

In [ ]:
axes = plt.subplots(2, 2, figsize=(10.5, 7.5), sharex=True, layout="constrained")[1]
for row, name in enumerate(wells):
    for column, side in enumerate(("health", "relapse")):
        panel = axes[row][column]
        block = sweep[(sweep["well"] == name) & (sweep["side"] == side)]
        inverse = 1.0 / block["sigma"] ** 2
        panel.semilogy(inverse, block["kramers"], marker="o", label="Kramers")
        panel.semilogy(inverse, block["mfpt_bottom_to_saddle"], marker="s", label="to the saddle")
        panel.semilogy(inverse, block["mfpt_bottom_to_bottom"], marker="^", label="well to well")
        panel.semilogy(
            inverse,
            block["simulated_mean"],
            marker="x",
            linestyle="none",
            color="black",
            label="simulated",
        )
        panel.set_title(f"{name}, {side} well")
        panel.set_xlabel("1 / sigma^2")
        panel.set_ylabel("mean exit time (weeks)")
axes[0][0].legend(fontsize="small")
plt.show()

The three analytic readings answer three different questions. Kramers is a well to well time
in the small noise limit. In the deep health well it sits above the exact first passage time
to the saddle by about a factor of two, because a walker that reaches the saddle still falls
back about half the time. The factor grows as a well gets shallower against the noise, since
the walker then comes back into the well more often: in the relapse well of the calibrated
potential, the shallowest of the four, it runs from about two at the smallest noise of the
sweep to about four at the largest, while the relapse well of the illustrative potential,
whose barrier is more than twice as high, only reaches about two and a half. The exact well to
well time is the one Kramers approximates, and over this range of noise the two stay within
about a fifth of one another; the formula is exact only as the noise goes to zero, which the
durations of the article do not reach. The simulated points follow the curve to the saddle,
which is the passage they were measured over.

In [ ]:
panel = plt.subplots(figsize=(7.0, 4.2), layout="constrained")[1]
for name in wells:
    for side in ("health", "relapse"):
        block = sweep[(sweep["well"] == name) & (sweep["side"] == side)]
        panel.plot(block["sigma"], block["simulated_cv"], marker="o", label=f"{name}, {side}")
panel.axhline(1.0, linestyle="--", color="grey")
panel.set_xlabel("sigma")
panel.set_ylabel("coefficient of variation of the exit time")
panel.set_ylim(0.0, 1.3)
panel.legend(fontsize="small")
plt.show()

An exponential exit time has a coefficient of variation of 1, and every point of the panel
sits near that line. That is not evidence that all four wells are exponential. With 300 paths
behind each point the coefficient of variation of a sample that really is exponential already
scatters by about 0.06, and the points span roughly 0.82 to 1.12, so the panel cannot tell
the deep health well from the shallow relapse one. What it does rule out is an exit time far
from exponential, which would sit well off the line at every noise. The difference between
the two wells shows instead in the regime table below and in the shape of the survival curves
beside it, where the shallow well keeps far more paths inside at short times than a
memoryless law would.

## Why the relapse well of the calibrated potential is not exponential

An exit time is exponential when leaving the well is a rare event, that is when the barrier
is high against the noise and the walker relaxes to the bottom of the well many times before
it escapes. The exponent `2 dV / sigma^2` is that comparison, and the Kramers prefactor is
the relaxation time the exit time has to be long against. At the calibrated parameters the
health well clears both, by a barrier of about twice the noise variance and an exit time of
about twenty relaxation times. The relapse well clears neither: its barrier sits below the
noise variance and its mean exit time is shorter than the relaxation time itself, so there is
no memoryless law behind the record it produces, however the article reads its histogram.

In [ ]:
barriers = calibrated.barriers()
regime_rows = []
for side, barrier in (("health", barriers.health), ("relapse", barriers.relapse)):
    regime_rows.append(
        (
            side,
            barrier,
            2.0 * barrier / calibrated_sigma**2,
            msrelapse.kramers_prefactor(calibrated, side),
            msrelapse.mfpt(calibrated, calibrated_sigma, side),
        )
    )

regime = pd.DataFrame(
    regime_rows,
    columns=["side", "barrier", "two_dV_over_sigma2", "prefactor_w", "mfpt_w"],
)
regime

In [ ]:
N_SHAPE_PATHS = 2000
shape_generator = np.random.default_rng(SEED)
shapes = {
    side: msrelapse.exit_times(
        calibrated, calibrated_sigma, side, N_SHAPE_PATHS, dt=DT, rng=shape_generator
    )
    for side in ("health", "relapse")
}

panel = plt.subplots(figsize=(7.0, 4.2), layout="constrained")[1]
grid = np.linspace(0.0, 5.0, 200)
panel.semilogy(grid, np.exp(-grid), color="black", linestyle="--", label="exponential")
for side, times in shapes.items():
    scaled = np.sort(times / times.mean())
    survival = 1.0 - np.arange(scaled.size) / scaled.size
    cv = times.std(ddof=1) / times.mean()
    panel.semilogy(scaled, survival, label=f"{side} well, cv {cv:.2f}")
panel.set_ylim(1e-3, 1.05)
panel.set_xlabel("exit time, in units of its own mean")
panel.set_ylabel("share of paths still inside the well")
panel.legend()
plt.show()

## The Brownian bridge correction

A threshold tested only at the grid points is crossed a step late, because a path that
crossed and came back between two samples is never seen. The error grows like the square root
of the step, and the correction moves the absorbing level towards the walker by
`0.5826 sigma sqrt(dt)`, the mean overshoot of a Brownian bridge past a level. The sweep
below is the same passage measured at six steps with the correction on and off, against the
exact mean first passage time.

In [ ]:
# More paths than the sweep above, because the bias this measures is a few
# percent and has to stand clear of the noise of the mean.
BRIDGE_PATHS = 6000
DT_GRID = np.array([0.005, 0.01, 0.02, 0.05, 0.1, 0.2])
bridge_sigma = msrelapse.PAPER.noise_amplitude.value
exact_relapse = msrelapse.mfpt(illustrative, bridge_sigma, "relapse")

# Here the seed is deliberately repeated rather than carried in a generator: the
# two series of each step start from the same noise stream, so what separates
# them is the correction and not a different draw.
bridge_rows = []
for dt in DT_GRID:
    for corrected in (True, False):
        times = msrelapse.exit_times(
            illustrative,
            bridge_sigma,
            "relapse",
            BRIDGE_PATHS,
            dt=dt,
            rng=SEED,
            bridge_correction=corrected,
        )
        bridge_rows.append((float(dt), corrected, float(times.mean())))

bridge = pd.DataFrame(bridge_rows, columns=["dt", "bridge_correction", "mean_w"])
bridge["relative_error"] = bridge["mean_w"] / exact_relapse - 1.0
bridge

In [ ]:
panel = plt.subplots(figsize=(7.0, 4.2), layout="constrained")[1]
for corrected in (True, False):
    block = bridge[bridge["bridge_correction"] == corrected]
    panel.plot(
        np.sqrt(block["dt"]),
        block["mean_w"],
        marker="o",
        label=f"bridge_correction={corrected}",
    )
panel.axhline(exact_relapse, linestyle="--", color="grey", label="exact mean first passage time")
panel.set_xlabel("sqrt(dt)")
panel.set_ylabel("mean exit time (weeks)")
panel.legend()
plt.show()

## What to take away

The uncorrected mean rises with the step, close to a straight line in the square root of it
while the step is short and flattening at the longest steps. With the correction the same
measurement sits on the exact value until the step is long enough for the Euler step itself
to matter, which is where the corrected series turns down. Everything else in this package
that reads a crossing off a grid, `exit_times` and `simulate_weekly` alike, carries the same
correction by default.

## How to cite

This notebook reproduces the article below, and the package it uses carries the same
reference on every result object. The text is what `msrelapse.citation()` prints.

```text
Bordi I, Umeton R, Ricigliano VAG, Annibali V, Mechelli R, Ristori G, Grassi F, Salvetti M, Sutera A. A mechanistic, stochastic model helps understand multiple sclerosis course and pathogenesis. International Journal of Genomics. 2013;2013:910321. doi:10.1155/2013/910321.

@article{bordi2013mechanistic,
  author    = {Bordi, Isabella and Umeton, Renato and Ricigliano, Vito A. G. and Annibali, Viviana and Mechelli, Rosella and Ristori, Giovanni and Grassi, Francesca and Salvetti, Marco and Sutera, Alfonso},
  title     = {A mechanistic, stochastic model helps understand multiple sclerosis course and pathogenesis},
  journal   = {International Journal of Genomics},
  volume    = {2013},
  pages     = {910321},
  year      = {2013},
  doi       = {10.1155/2013/910321},
  publisher = {Hindawi}
}

% The DOI below is a placeholder: the archive DOI is minted at the first release.
@software{msrelapse,
  author  = {Umeton, Renato},
  title   = {msrelapse: a reference implementation of the Bordi et al. 2013 double well model of multiple sclerosis},
  year    = {2026},
  version = {0.1.0},
  url     = {https://github.com/renato-umeton/multiple-sclerosis-modeling-bordi},
  doi     = {10.5281/zenodo.XXXXXXX}
}
```